# LLM Text Annotation: Judging and Improving Output Quality with Local Fine-Tuning

### Setup

Every model run was done once locally and saved. It reads outputs from cache.

In [1]:
import json, sys
from pathlib import Path

sys.path.insert(0, ".")

import numpy as np
import pandas as pd

from src import config as cfg, data, viz
from src.annotate import annotate, cached_records
from src.prompts import RUBRIC

pd.set_option("display.max_colwidth", 90)
viz.style()

for k, v in cfg.summary().items():
    print(f"{k:>14}: {v}")

         model: unsloth/Qwen3-1.7B-unsloth-bnb-4bit
          seed: 20260810
      eval set: 400 comments (200 per class)
     probe set: 300 comments
 saved answers: 2 model runs in cache/


## 1. Annotation is measurement

The SFU Opinion and Comments Corpus collects reader comments from *The Globe and Mail*. Suppose we want to know whether comment sections get less constructive during election campaigns. Before we can answer that, every comment needs a label, and there are far too many to read.

In [2]:
articles = pd.read_csv(cfg.ARTICLES_CSV, usecols=["article_id", "title", "published_date", "ncomments"])
gold = data.load_gold()

print(f"articles in the corpus : {len(articles):,}")
print(f"comments on them       : {articles.ncomments.sum():,.0f}")
print(f"hand-labelled by humans: {len(gold):,}")

articles in the corpus : 10,339
comments on them       : 663,173
hand-labelled by humans: 1,043


Someone labelled 1,043 of them by hand. That set is our answer key, and the rest of the notebook is about whether a model can extend it to the other 662,000.

A comment counts as **constructive** if it tries to add something: a specific point, evidence, a personal experience, a proposed solution. A comment is **not constructive** if it is only an insult, a one-line dismissal, or sarcasm with nothing behind it. Here is one of each.

In [3]:
for label in ("yes", "no"):
    row = gold[gold.is_constructive == label].iloc[1]
    print(f"[constructive: {label}]  {row.comment_text[:290]}\n")

[constructive: yes]  Everyone is still missing the point of what the Apple Watch is:It's fashion. It is by definition of no utility. Watches have been more fashion than function for as long as they've been worn. Anyone remember paying $50 for a Swatch that cost $2 to make? The Apple Watch actually offers quite

[constructive: no]  You may be using a blackberry. I'm still using a gooseberry.



Two features of the answer key matter later. It is nearly balanced, and it comes from only a handful of articles, so it is a narrow slice of the corpus.

In [4]:
print(gold.is_constructive.value_counts().to_string())
print(f"\ndrawn from {gold.article_id.nunique()} articles")

is_constructive
yes    554
no     489

drawn from 13 articles


We split it once, now. The evaluation half is scored twice in what follows, once per method, but it never *decides* anything: no setting and no model is ever chosen by looking at it. The probe set is used later to look inside the model, and is kept separate so that nothing we inspect during training is also something we score on.

In [5]:
eval_set, probe_set = data.gold_splits()
print(f"evaluation set: {len(eval_set)} comments, {(eval_set.is_constructive == 'yes').sum()} constructive")
print(f"probe set     : {len(probe_set)} comments, held apart from the evaluation set")

evaluation set: 400 comments, 200 constructive
probe set     : 300 comments, held apart from the evaluation set


## 2. Asking a model to annotate

The model is **Qwen3-1.7B**, an open-weights model small enough to run locally. It is stored at 4-bit precision, which means each weight is squeezed into a quarter of the usual space. That shrinks it to about 1.3 GB and is the reason it fits on a consumer graphics card at all.

Zero-shot means we describe the task and ask, with no examples and no training.

In [6]:
print(RUBRIC)

You are annotating reader comments from a Canadian news website.

A comment is CONSTRUCTIVE if it tries to add something to the conversation: it makes a specific point, gives evidence or a personal experience, offers a solution, or engages with the article's argument.

A comment is NOT CONSTRUCTIVE if it is only an insult, a one-line dismissal, sarcasm with no substance, off-topic ranting, or an unsupported assertion.

Comment:
"""{comment}"""

Reply in exactly this format and nothing else:
LABEL: yes
REASON: <one short sentence>


Each comment went through that prompt once, and the answer was saved under a key made from the model, the prompt and the comment. `annotate` reads those answers back. Editing the prompt changes the key, so it would find nothing saved.

In [7]:
zero_shot = annotate(eval_set.comment_counter, RUBRIC)

pd.DataFrame({"comment": eval_set.comment_text.str[:70],
              "human": eval_set.is_constructive,
              "model": zero_shot}).head(8)

,comment,human,model
0,Plenty. Ever been to Vancouver? There are condo boards that refuse to,no,no
1,Great piece! Thanks.,no,no
2,Tail wagging the dog,no,no
3,"Apparently, trying not to offend and be politically correct just doesn",no,no
4,The Belgian jihadis come mainly from the Rif mountains in northern Mor,no,no
5,Just keep whining - next thing that will happen is a ban on foreign bu,no,no
6,Why does the Globe and Mail even publish such a simplistic and accusat,no,no
7,"ROTFLMAO - hellloooo , no insurance, tax fraud,,, great starts, not to",no,no


It is worth looking at what the model actually wrote, not just the label we parsed out of it. Asking for a fixed format is what makes the output usable as data.

In [8]:
records = cached_records(RUBRIC)
for cid in eval_set.comment_counter.head(3):
    print(records[cid]["raw"], "\n" + "-" * 60)

LABEL: no
REASON: The comment is not constructive because it is an insult and lacks specific evidence or a personal experience. 
------------------------------------------------------------
LABEL: no
REASON: The comment is a one-line dismissal and does not add anything to the conversation. 
------------------------------------------------------------
LABEL: no
REASON: The comment is a sarcastic and unsupported assertion. 
------------------------------------------------------------


## 3. Judging the labels

Accuracy is the share the model got right. **Cohen's kappa** subtracts the agreement you would expect from chance, so 0 means "no better than guessing" and 1 means perfect.

In [9]:
zs = data.metrics(eval_set.is_constructive, zero_shot)
pd.Series(zs).round(3).to_frame("zero-shot")

,zero-shot
accuracy,0.730
f1,0.649
cohen_kappa,0.460
predicted_yes,0.270
unparsed,0.000


Kappa around 0.46 counts as poor agreement. The confusion matrix shows what shape the errors take.

In [10]:
data.confusion(eval_set.is_constructive, zero_shot)

,model said constructive,model said not constructive,share the model agreed
humans said constructive,100,100,0.50
humans said not constructive,8,192,0.96


Look at the last column. The model agrees with the humans on 96% of the comments they called not constructive, and on half of the ones they called constructive. It is not randomly unreliable. It has one habit, saying "not constructive" too much. This is the main cause of all of it's errors.

In [11]:
disagree = eval_set.assign(model=zero_shot)
missed = disagree[(disagree.is_constructive == "yes") & (disagree.model == "no")]
print(f"{len(missed)} constructive comments called not constructive. Two of them:\n")
for t in missed.comment_text.head(2):
    print(" ", t[:250], "\n")

100 constructive comments called not constructive. Two of them:

  Simpson is having his little joke by encouraging the NDP to play with this expensive hand grenade. If it weren't for the poisonous potential it introduces to the federal/provincial arena if would be an interesting experiment. Unfortunately if you wav 

  Paul The first world relates to per capita income, and now how that income was achieved. Take away the loot the the UK, France, Belgium, Spain, Portugal, etc stole from Asia, Africa and Latin America, and they would all be third world countries too.  



Before deciding whether kappa 0.46 is bad, we need something to compare it against. We will use the **human ceiling**. About a fifth of the answer key was labelled a second time by an expert, and the crowd and the expert do not always agree either.

In [12]:
from sklearn.metrics import cohen_kappa_score

has_expert = gold[gold.expert_is_constructive.notna()]
crowd = (has_expert.is_constructive.str.lower() == "yes").astype(int)
expert = (has_expert.expert_is_constructive.str.lower() == "yes").astype(int)

print(f"{len(has_expert)} comments were labelled twice")
print(f"crowd and expert agree on {(crowd == expert).mean():.1%} of them")
print(f"as kappa, that is {cohen_kappa_score(crowd, expert):.3f}")

214 comments were labelled twice
crowd and expert agree on 78.5% of them
as kappa, that is 0.583


## 4. Fine-tuning with QLoRA

Prompting can only rearrange what the model already does. Fine-tuning changes the model itself, by showing it labelled examples and adjusting its weights when it gets them wrong.

Doing that the ordinary way means updating all 1.7 billion weights, which needs far more memory than we have. **QLoRA** avoids it with two tricks:

- **Quantization**: the original weights stay frozen at 4 bits and are never updated.
- **LoRA**: we bolt small trainable matrices onto the frozen model and train only those.

The result is that under 2% of the model is trainable, and what we save at the end is one small file of adapter weights instead of a second copy of a 1.7 billion parameter model.

In [13]:
LORA = dict(
    r=16,                      # size of the bolted-on matrices
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",     # attention
                    "gate_proj", "up_proj", "down_proj"],       # and the MLP
)

TRAIN_ARGS = dict(
    per_device_train_batch_size=2,     # 8 GB of VRAM does not allow more
    gradient_accumulation_steps=8,     # so accumulate to an effective batch of 16
    max_steps=cfg.TRAIN_STEPS,
    learning_rate=2e-4,
    fp16=True, bf16=False,             # Turing GPUs have no native bf16
    gradient_checkpointing=True,       # trade compute for memory
)

Each training example is the same prompt we have been using, paired with the human label. The loss is computed **only on the label**, so the model is never rewarded for reproducing the prompt back to us.
That run took about 24 minutes. Its log was saved, and every figure below comes from it.

In [14]:
train_log = json.loads((cfg.ARTIFACTS / "train_log.json").read_text())

print(train_log["trainable"])
print(f"{train_log['n_train']} training examples, {cfg.TRAIN_STEPS} steps")
print(f"{train_log['train_seconds'] / 60:.1f} minutes at {train_log['seconds_per_step']} s/step")
print(f"peak VRAM {train_log['peak_vram_mib']} MiB, final loss {train_log['final_loss']}")

17,432,576 trainable of 1,052,238,848 (1.66%)
2000 training examples, 200 steps
23.9 minutes at 7.18 s/step
peak VRAM 4899 MiB, final loss 0.2068


## 5. When did training stop helping?

A falling loss tells us the model fitted the training data. It does not tell us whether the labels got better. So during training we paused every 20 steps and pushed the same 300 probe comments through the model, recording the answer it would have given and its internal state at that moment.

Those 300 comments are part of the answer key and were never trained on, so scoring them at each checkpoint says honestly what the training bought.

In [15]:
checkpoints = dict(np.load(cfg.ARTIFACTS / "probe_checkpoints.npz"))
curve = viz.learning_curve(checkpoints, cfg.MEDIA / "learning.png")

pd.DataFrame({"step": curve["steps"],
              "agreement (kappa)": curve["kappa"].round(3),
              "probe accuracy": curve["probe_accuracy"].round(3)})

,step,agreement (kappa),probe accuracy
0,0,0.373,0.837
1,20,0.620,0.853
2,40,0.520,0.850
3,60,0.667,0.857
4,80,0.673,0.860
5,100,0.660,0.860
6,120,0.667,0.860
7,140,0.680,0.863
8,160,0.687,0.860
9,180,0.693,0.863


![Left: agreement with the human labels on 300 held-out comments at each training checkpoint, with a shaded 95% range. Right: how accurately a straight line can recover the human label from the model's internal state, which stays flat.](media/learning.png)

Almost all of the gain arrives in the first 60 steps. Agreement climbs from 0.37 to 0.67 and then stops. The remaining 140 steps add 0.03, and with 300 comments the shaded range cannot resolve a difference that small. We could have trained for a third as long and finished with the same annotator.

The right panel asks something else. If we take the model's internal state and try to recover the human label from it with a straight line, how well does that work? Barely better at the end than at the start, 0.84 to 0.86. The two groups were already separable inside the model before we trained anything. Training changed the model's answers. It did not change what the model could tell apart.

## 6. Does it label better?

The same 400 comments, the same prompt, the model we just trained. Nothing about this model was chosen by looking at these comments.

In [16]:
fine_tuned = annotate(eval_set.comment_counter, RUBRIC, adapter=cfg.ADAPTER)
truth = eval_set.is_constructive

wrong_before, wrong_after = viz.comparison_grid(truth, zero_shot, fine_tuned,
                                                cfg.MEDIA / "before_after.png")
fixed = sum(z != t and f == t for t, z, f in zip(truth, zero_shot, fine_tuned))
broke = sum(z == t and f != t for t, z, f in zip(truth, zero_shot, fine_tuned))

print(f"zero-shot got {wrong_before} of {len(truth)} wrong; fine-tuned got {wrong_after} wrong")
print(f"fine-tuning fixed {fixed} comments and broke {broke}")

zero-shot got 108 of 400 wrong; fine-tuned got 47 wrong
fine-tuning fixed 72 comments and broke 11


![Two grids of 400 squares, one per evaluation comment, with the squares the model got wrong picked out. Comments humans called constructive sit above the dividing line. The zero-shot grid is heavily marked across its top half; the fine-tuned grid is mostly clear.](media/before_after.png)

Each square is one comment, and the ones above the line are those humans called constructive.

The zero-shot grid is the bias from section 3 made visible. The top half is covered in errors and the bottom half is nearly clean: the model was reliable at spotting what is *not* constructive and unreliable at everything else. After fine-tuning the top half mostly clears.

In [17]:
pd.DataFrame({"zero-shot": data.metrics(truth, zero_shot),
              "fine-tuned": data.metrics(truth, fine_tuned)}).round(3)

,zero-shot,fine-tuned
accuracy,0.730,0.882
f1,0.649,0.875
cohen_kappa,0.460,0.765
predicted_yes,0.270,0.438
unparsed,0.000,0.000


Fine-tuning works. Kappa goes from 0.46 to 0.77, and the model now calls 44% of comments constructive against a true rate of 50%, so the bias from section 3 is gone.

The crowd and the expert agreed with each other at kappa 0.58, so the model now agrees with the crowd's labels more closely than the expert does. That does not make it better than the expert. It has learned how these particular annotators used the word.

In [18]:
# a comment the zero-shot model got wrong and the fine-tuned one got right
flipped = eval_set[[z != t and f == t for t, z, f in zip(truth, zero_shot, fine_tuned)]]
row = flipped.iloc[1]
saved_before = cached_records(RUBRIC)[row.comment_counter]["raw"]
saved_after = cached_records(RUBRIC, adapter=cfg.ADAPTER)[row.comment_counter]["raw"]

print(row.comment_text[:400])
print()
print("humans     ->", row.is_constructive)
print("zero-shot  ->", " ".join(saved_before.split()))
print("fine-tuned ->", " ".join(saved_after.split()))

Paul The first world relates to per capita income, and now how that income was achieved. Take away the loot the the UK, France, Belgium, Spain, Portugal, etc stole from Asia, Africa and Latin America, and they would all be third world countries too. Don't believe me? Look up the kohinoor diamond - where it was, and where it is today.

humans     -> yes
zero-shot  -> LABEL: no REASON: The comment is not constructive because it is an unsupported assertion and contains a factual error (the Kohinoor diamond is actually in the British Museum, not in India).
fine-tuned -> LABEL: yes


Read the zero-shot reason. It applies the rubric, calling the comment an unsupported assertion, and then reaches past it to dispute whether the comment is factually right. That is a different question from the one we asked.

Fine-tuning also cost us something. The zero-shot model explains itself, because we asked it for a reason and it had never been told otherwise. The fine-tuned model was trained on bare labels, so bare labels are what it gives. It became more accurate and less inspectable at the same time, and if the reasons matter to your research you would train on labels that carry one.

## 7. Does it work on new data?

Everything so far lives in one corpus of Canadian news comments. A label is only useful if it means the same thing somewhere else, so we move to a different kind of argument entirely.

The **Winning Arguments** corpus collects threads from Reddit's *ChangeMyView*, where someone states a view and other people try to change it. If the original poster is persuaded they award a delta. Tan et al. paired every successful argument with an unsuccessful one from the same discussion, which controls for the topic and for who was arguing.

In [19]:
cmv_set = pd.read_csv(cfg.CMV_CSV)
build = json.loads((cfg.ARTIFACTS / "cmv_build.json").read_text())

print(f"matched pairs available   : {build['pairs_available']:,}")
print(f"used here                 : {len(cmv_set) // 2} pairs, {len(cmv_set)} comments")
print(f"dropped for naming a delta: {build['dropped_award_text']}")
print()
print(cmv_set.persuasive.value_counts().to_string())

matched pairs available   : 3,981
used here                 : 100 pairs, 200 comments
dropped for naming a delta: 32

persuasive
no     100
yes    100


Note what we are asking. The model was trained to spot **constructive** comments, judged by crowdworkers on news articles. We are testing it on **persuasive** comments, judged by whoever happened to be arguing. Those are not the same idea.

In [20]:
data.length_report(cmv_set, gold)

,set,n,median chars,p90 chars
0,CMV persuasive,100,1173,2590
1,CMV not persuasive,100,843,1914
2,SOCC constructive,554,433,1025
3,SOCC not constructive,489,111,243


Reddit arguments are long, and both classes are long. If our annotator has quietly learned "long means constructive", it should call almost everything here constructive.

In [21]:
cmv_scores = {}
for name, adapter in (("zero-shot", None), ("fine-tuned", cfg.ADAPTER)):
    labels = annotate(cmv_set.comment_id, RUBRIC, adapter=adapter)
    cmv_scores[name] = data.metrics(cmv_set.persuasive, labels)

pd.DataFrame(cmv_scores).round(3)

,zero-shot,fine-tuned
accuracy,0.515,0.500
f1,0.628,0.667
cohen_kappa,0.030,0.000
predicted_yes,0.805,1.000
unparsed,0.000,0.000


Both sit at kappa 0, which is chance. The fine-tuned model calls **every single** Reddit comment constructive.

WE NEED NEW DATA OR MODEL OR A BETTER IDEA




### Data and licences

- **SOCC** and its constructiveness subset: Kolhatkar, V., H. Wu, L. Cavasso, E. Francis, K. Shukla and M. Taboada (2020). The SFU Opinion and Comments Corpus: A corpus for the analysis of online news comments. *Corpus Pragmatics* 4(2), 155-190. https://doi.org/10.1007/s41701-019-00065-w Licensed CC BY-NC-SA 4.0.
- **C3**: Kolhatkar, V., N. Thain, J. Sorensen, L. Dixon and M. Taboada (2020). *C3: The Constructive Comments Corpus.* Jigsaw and Simon Fraser University. DOI: 10.25314/ea49062a-5cf6-4403-9918-539e15fd7b52 Licensed CC BY-NC 4.0.
- **Winning Arguments**: Tan, C., V. Niculae, C. Danescu-Niculescu-Mizil and L. Lee (2016). Winning arguments: Interaction dynamics and persuasion strategies in good-faith online discussions. *Proceedings of the 25th International Conference on World Wide Web (WWW '16)*, 613-624. https://arxiv.org/abs/1602.01103 Distributed in ConvoKit.
- **Model**: Qwen3-1.7B, Apache 2.0, 4-bit build by Unsloth.

Both comment corpora are non-commercial licences, so anything built from them inherits that restriction.

### References

- Fang, Q., J. Garcia Bernardo and E-J. van Kesteren (2026). *A Methodological Guide on Using Large Language Models for Text Annotation in the Social Sciences and Humanities with Python and R.* https://arxiv.org/abs/2604.09638
- Dettmers, T., Pagnoni, A., Holtzman, A., & Zettlemoyer, L. (2023). *QLoRA: Efficient finetuning of quantized LLMs.* https://arxiv.org/abs/2305.14314
- Hu, E., et al. (2021). *LoRA: Low-rank adaptation of large language models.* https://arxiv.org/abs/2106.09685
- Krippendorff, K. (2018). *Content Analysis: An Introduction to Its Methodology.* SAGE. On treating annotation as measurement.
- Danescu-Niculescu-Mizil, C., et al. ConvoKit. https://convokit.cornell.edu/